# 03. Maskable PPO: architecture and training workflow

UTS 43008 · Assignment 3 · Project 12 · Dihedrals

Does training with rotated and reflected coordinate views improve generalisation when the
learner, layouts and actual training budget are otherwise matched?

P is standard Maskable PPO. PA is the same method with D4 view augmentation at reset. Our custom
work covers the game, RL interface, feature extractor, transformations and experiment workflow.
PPO itself is supplied by SB3 Contrib.

Full training is disabled here. Run the short recovery lab in **03b** before the long experiments.

In [ ]:
from pathlib import Path
import os
import sys
import json
import uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

candidate = Path(os.environ.get("BASE_SIEGE_ROOT", Path.cwd())).expanduser()
ROOT = next((p for p in [candidate, *candidate.parents] if (p / "env/siege_env.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the complete project, or set BASE_SIEGE_ROOT to its folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from env import config as C
from env.siege_env import SiegeEnv, describe_action, deployment_action
from env.render import draw_snapshot
from env.generator import generate_base
from experiments.provenance import runtime_manifest, sha256
RUNS = Path(os.environ.get("BASE_SIEGE_RUNS_DIR", ROOT / "runs")).expanduser().resolve()
OUTPUT = ROOT / "artifacts" / "notebook-sessions" / uuid.uuid4().hex[:12]
OUTPUT.mkdir(parents=True, exist_ok=False)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 12, "axes.labelsize": 10, "font.size": 10})
def save_figure(name):
    plt.savefig(OUTPUT / f"{name}.png", dpi=180, bbox_inches="tight")
    plt.show()
print(f"Project: {ROOT.name} | Python: {sys.version.split()[0]}")
print(f"This notebook's outputs: {OUTPUT}")

## 1. Observation to decision

The CNN extracts features from eight board layers. A linear layer processes the nine summary
values. Their outputs are concatenated into 256 shared features. Separate heads estimate action
preferences and state value.

The action head has 246 outputs. Illegal actions are masked before sampling. The value head
estimates expected future return, not the current destruction percentage.

In [ ]:
import torch
from train.features import SiegeFeatures
from train.train_ppo import TrainingSettings, phase_budgets, make_env, train, resume
from train.checkpoints import RecoverablePPO
from stable_baselines3.common.vec_env import DummyVecEnv
from dataclasses import asdict

torch.set_num_threads(1)
board = SiegeEnv()
observation, _ = board.reset(seed=0)
extractor = SiegeFeatures(board.observation_space)
display(extractor)
with torch.no_grad():
    features = extractor({k: torch.as_tensor(v).unsqueeze(0) for k, v in observation.items()})
print("Feature output shape:", tuple(features.shape))
print("Feature extractor parameters:", sum(p.numel() for p in extractor.parameters()))
board.close()

In [ ]:
display(pd.DataFrame([
    ["Grid input", "8 × 16 × 16"], ["Conv 3×3, stride 2, ReLU", "32 × 8 × 8"],
    ["Conv 3×3, stride 2, ReLU", "64 × 4 × 4"], ["Flatten", "1,024"],
    ["Vector linear + ReLU", "9 → 32"], ["Concatenate + linear + ReLU", "1,056 → 256"],
    ["Policy head", "256 → 128 → 246"], ["Value head", "256 → 128 → 1"]
], columns=["operation", "dimensions"]))

## 2. What PPO changes

PPO collects actions, rewards, old action probabilities and value estimates. It computes
advantages: was an outcome better or worse than expected? It updates the policy using a clipped
probability ratio, discouraging large changes from one batch.

Positive advantage encourages an action; negative advantage discourages it. Clipping is a
training objective, not a strict guarantee on every action probability.

In [ ]:
ratio = np.linspace(.5, 1.5, 201)
epsilon = .2
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for advantage, ax in zip([1., -1.], axes):
    ordinary = ratio * advantage
    clipped = np.clip(ratio, 1-epsilon, 1+epsilon) * advantage
    ax.plot(ratio, ordinary, "--", label="Unclipped")
    ax.plot(ratio, np.minimum(ordinary, clipped), label="PPO clipped")
    ax.axvline(1, color="grey", alpha=.4)
    ax.set(title=f"Advantage = {advantage:+.0f}", xlabel="New / old action probability", ylabel="Objective to maximise")
    ax.legend()
plt.tight_layout()
save_figure("ppo-clipping-example")

## 3. A worked advantage calculation

The one-step residual is reward + gamma × next_value − current_value. Generalised advantage
estimation combines residuals backwards through the rollout using gamma and lambda. Terminal
states have no future value. Truncated transitions may need bootstrapping; the library uses
the environment's ending signals.

These numbers illustrate the mathematics. They are not training results.

In [ ]:
rewards = np.array([0.0, .2, .5])
values = np.array([.1, .15, .3, 0.0])
residuals = rewards + C.PPO_GAMMA * values[1:] - values[:-1]
advantages = np.zeros_like(rewards)
carry = 0.0
for i in reversed(range(len(rewards))):
    carry = residuals[i] + C.PPO_GAMMA * C.PPO_GAE_LAMBDA * carry
    advantages[i] = carry
display(pd.DataFrame({"reward": rewards, "value": values[:-1], "TD residual": residuals,
    "GAE advantage": advantages, "return target": advantages + values[:-1]}))

## 4. Inspect a masked prediction

This initialised model has not learned. We only check that the selected action is legal.
Masks must be supplied during evaluation as well as training.

In [ ]:
vector = DummyVecEnv([make_env])
initial = RecoverablePPO("MultiInputPolicy", vector, seed=0, device="cpu", n_steps=32,
    batch_size=32, n_epochs=2, policy_kwargs={"features_extractor_class": SiegeFeatures,
    "normalize_images": False, "net_arch": {"pi": [128], "vf": [128]}})
observation = vector.reset()
mask = np.stack(vector.env_method("action_masks"))
action, _ = initial.predict(observation, action_masks=mask, deterministic=True)
print("Initial legal choice:", describe_action(int(action[0])))
print("Legal actions:", int(mask.sum()))
assert mask[0, action[0]]
vector.close()

## 5. What D4 augmentation does

A square has eight rotation/reflection views. PA samples one at reset and transforms the board,
deployment coordinates and mask together. Shopping actions and summary values stay unchanged.

The physical battle remains in its original coordinates. The simulator's coordinate tie-breaks
are not claimed to be rotation-equivariant. A separate view RNG prevents augmentation from
changing the sequence of sampled layouts.

In [ ]:
from env.symmetry import action_permutation, transform_observation
board = SiegeEnv(base_seed=42, motifs=("closed_ring",))
observation, _ = board.reset(seed=0)
fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
mapping = []
for view, ax in enumerate(axes.flat):
    permutation = action_permutation(view)
    assert np.array_equal(np.sort(permutation), np.arange(C.N_ACTIONS))
    transformed = transform_observation(observation, view)
    ax.imshow(transformed["grid"][:5].sum(axis=0), cmap="cividis")
    ax.set_title(f"View {view}")
    ax.set_xticks([])
    ax.set_yticks([])
    mapping.append({"view": view, "physical_action": 12, "view_action": int(permutation[12])})
plt.tight_layout()
save_figure("d4-views")
display(pd.DataFrame(mapping))
board.close()

## 6. Curriculum and matched budgets

Placement training supplies the heuristic army. Joint training starts new shopping episodes and
learns purchases as well as deployment. Policy and optimiser continue across the phase boundary.

Revision 2 directly requests 32,768 + 229,376 = 262,144 transitions. These are four placement
rollouts and twenty-eight joint rollouts with four workers and 2,048 steps per worker.
The old draft requested 250,000, which rounded to this count. Both methods receive equal budgets.

In [ ]:
full_settings = TrainingSettings(mode="full", device="cuda",
    placement_steps=C.CURRICULUM_PLACEMENT_TIMESTEPS, joint_steps=C.CURRICULUM_JOINT_TIMESTEPS,
    n_envs=C.N_TRAINING_ENVS, rollout_steps=C.PPO_N_STEPS, batch_size=C.PPO_BATCH_SIZE, epochs=C.PPO_N_EPOCHS)
display(pd.Series(asdict(full_settings), name="full_run_settings"))
display(pd.Series(phase_budgets(full_settings), name="actual_budget"))
display(pd.Series({name: getattr(C, name) for name in ["PPO_LEARNING_RATE", "PPO_GAMMA",
    "PPO_GAE_LAMBDA", "PPO_ENT_COEF", "PPO_N_EPOCHS", "PPO_BATCH_SIZE"]}, name="learning_parameters"))

## 7. Recovery is part of the training loop

After each completed rollout update, the trainer saves model and Adam optimiser, worker games,
RNG states, last observations, phase and committed history. Each generation has checksums.
The latest pointer is replaced only after the complete generation is written.

The exported model is the fixed-budget endpoint, not a checkpoint selected using final scores.
Development checks run at the first completed update at or after each 25,000-transition interval.
They use up to four spread layouts from each of three development structures across four views:
48 episodes when the panel is full. An initial check runs before learning. These diagnostics
are not the complete final evaluation. Pool data is referenced from verified run inputs rather
than copied into each worker checkpoint.

## 8. Optional short software run

This switch runs tiny CPU learners only. It cannot launch full training. New output folders
protect existing evidence. Smoke policies are not expected to play well.

In [ ]:
RUN_SOFTWARE_SMOKE = False
if RUN_SOFTWARE_SMOKE:
    for condition in ("P", "PA"):
        report = train(TrainingSettings(condition=condition), OUTPUT / f"smoke-{condition}")
        print(condition, report["status"], report["actual_steps"])
else:
    print("Smoke training not requested. Notebook 03b contains the recovery exercise.")

## 9. Prepare the study first

Build the full filtered pools. Run the separate 57,344-step pilot pair at learner seed 100, evaluate
it and record the team's evidence-based review. Registration verifies that review. The default
main study uses ten learner seeds per condition: 20 learners and 5,242,880 transitions.

See [Team master](../docs/TEAM_MASTER.md) and [Runbook](../docs/RUNBOOK.md). The next cell constructs
the study-controller command but does not execute it. Source changes after registration require
a new registration and matching fresh runs; the controller does not access final layouts.

In [ ]:
import shlex
full_command = [sys.executable, "-m", "experiments.study", "run", "--confirm-long-run",
    "--device", "cuda", "--registration", str(ROOT / "experiments/registration-v2.json"),
    "--out", str(RUNS / "study-v2")]
print(shlex.join(full_command))
print("Command only. Full training has NOT started.")

### Pilot budget and runtime review

The pilot uses the full network, optimiser and four-worker rollout configuration, with a shorter
joint phase. It is not included among the main learners. The command-line pilot review estimates
main-training hours from measured wall time and checks that the development evaluation belongs
to those pilots. The team still needs to assess learning curves and failures.

No runtime below is an observed SageMaker measurement. The table shows only the planned work.

In [ ]:
from train.train_ppo import study_settings
display(pd.DataFrame([
    {"stage": "pilot pair", "learners": 2, "steps_each": phase_budgets(study_settings(mode="pilot"))["total_actual"]},
    {"stage": "main study", "learners": 2 * len(C.STUDY_LEARNER_SEEDS),
     "steps_each": phase_budgets(study_settings())["total_actual"]},
]).assign(total_steps=lambda frame: frame.learners * frame.steps_each))

## 10. Inspect a real history

Select an existing run explicitly. After recovery, canonical episode history matches the
committed model state. Previous uncommitted history is archived separately. Compare returns
only when their reward definitions match.

In [ ]:
RUN_TO_INSPECT = None
if RUN_TO_INSPECT is not None:
    selected = Path(RUN_TO_INSPECT)
    manifest = json.loads((selected / "manifest.json").read_text())
    display(pd.Series({k: manifest.get(k) for k in ["status", "actual_steps", "current_phase", "evidence_scope"]}))
    history_file = selected / "episodes.jsonl"
    if history_file.exists() and history_file.stat().st_size:
        history = pd.read_json(history_file, lines=True)
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
        for column, ax in zip(["return", "destruction"], axes):
            ax.scatter(history.timesteps, history[column], s=8, alpha=.35, label="Episode")
            ax.plot(history.timesteps, history[column].rolling(20, min_periods=1).mean(), label="20-episode mean")
            ax.set(xlabel="Environment transitions", ylabel=column.title())
            ax.legend()
        plt.tight_layout()
        save_figure("training-history")
else:
    print("No run selected. No learning curve has been invented.")

## 11. Questions to explain

Why mask actions rather than penalising invalid moves? Why keep P and PA otherwise identical?
Why does recovery need more than weights? Why use independent learner seeds for uncertainty?
Why select the final model by budget rather than its final-test score?

Primary references: [Maskable PPO](https://sb3-contrib.readthedocs.io/en/master/modules/ppo_mask.html)
and [SB3 save format](https://stable-baselines3.readthedocs.io/en/master/guide/save_format.html).